# 1. Quick Reference & FAQ - Transformers with Keras

## Quick Concept Reference

### One-Sentence Definitions

| Concept | Definition |
|---------|-----------|
| **Embedding** | Converts word numbers into meaningful vector representations |
| **LSTM** | Neural network that reads sequences and remembers important information |
| **Self-Attention** | Mechanism to determine which parts of input are relevant for each output |
| **Encoder** | Part that reads and understands the input language (English) |
| **Decoder** | Part that generates output in target language (Spanish) using encoder context |
| **Softmax** | Converts numbers into probabilities that sum to 1 |
| **One-hot Encoding** | Vector with single 1 and rest 0s (e.g., [0,1,0,0]) |
| **Tokenization** | Converting text into numbers |
| **Padding** | Adding zeros to make sequences same length |
| **Batch** | Group of samples processed together |

---

## Quick Code Reference

### Common Commands

```python
# Data preparation
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Padding
from keras.preprocessing.sequence import pad_sequences
padded = pad_sequences(sequences, maxlen=10, padding='post')

# Building model
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding
model = Model(inputs=in_layer, outputs=out_layer)

# Compiling
model.compile(optimizer='adam', loss='categorical_crossentropy')

# Training
history = model.fit(X_train, y_train, epochs=100, batch_size=16)

# Plotting
import matplotlib.pyplot as plt
plt.plot(history.history['loss'])
plt.show()
```

---

## Frequently Asked Questions

### Q1: Why do we add "startseq" and "endseq"?

**A:** These special tokens teach the decoder:
- **startseq**: "Begin translation now"
- **endseq**: "Stop translating, you're done"

Without these, the model doesn't know when to start and stop.

**Example:**
```
Without markers:
Input: [1, 2, 3]
Target: [4, 5, 6]
Problem: Decoder doesn't know when to stop!

With markers:
Input: [startseq, 1, 2, 3, endseq]
Target: [startseq, 4, 5, 6, endseq]
Solution: endseq tells decoder to stop
```

---

### Q2: Why do we shift the decoder target sequence?

**A:** So the decoder learns to predict the NEXT word.

**Example:**
```
Sentence: "Hello world goodbye"
Tokens: [1, 2, 3]

Without shift (WRONG):
Decoder sees: [1, 2, 3]
Tries to predict: [1, 2, 3]
Problem: Can't predict past tense!

With shift (CORRECT):
Decoder sees: [1, 2]
Tries to predict: [2, 3]
Now it predicts next word!
(Input: [word1, word2], Output: [word2, word3])
```

---

### Q3: Why is vocab_size + 1?

**A:** The +1 is for **unknown words** (OOV - Out of Vocabulary).

**Example:**
```
Training words: {"hello": 1, "world": 2, "!": 3}
Vocab size = 3 + 1 = 4

Index 1: "hello"
Index 2: "world"
Index 3: "!"
Index 4: [Unknown] ← words not in training data

During inference, if we see "xyz" (unknown word):
Tokenizer outputs: 4 (the unknown token)
Embedding layer: recognizes index 4 = unknown word
```

---

### Q4: What does "return_sequences=True" do in LSTM?

**A:** Controls whether LSTM returns output for each word or just final word.

**Example:**
```
Sentence: "Hello how are"
Tokens: [1, 2, 3]

LSTM with return_sequences=False:
└─ Returns only final output: [h3]  (1 vector)
└─ Problem: loses information about early words

LSTM with return_sequences=True:
└─ Returns all outputs: [h1, h2, h3]  (3 vectors)
└─ Good for: attention, word-level predictions
```

---

### Q5: What does "return_state=True" do in LSTM?

**A:** Makes LSTM also return hidden and cell states (memory).

**Example:**
```
LSTM with return_state=True:
Returns 3 things:
1. outputs: All hidden states [h1, h2, h3]
2. state_h: Final hidden state [h3]
3. state_c: Final cell state [c3]

Why needed?
- Outputs: Attend to each word
- state_h, state_c: Pass to decoder as initial state
  (Decoder "starts from" encoder's understanding)
```

---

### Q6: What's the difference between hidden state (h) and cell state (c)?

**A:** Both are memory, but serve different purposes.

**Hidden State (h):**
- Short-term memory (what happened recently)
- Directly affects current output
- Used to generate predictions

**Cell State (c):**
- Long-term memory (persistent patterns)
- Passes through LSTM cells
- More stable, less noisy

**Analogy:**
- Hidden state: "Short-term memory" (remember last sentence)
- Cell state: "Long-term memory" (remember language patterns)

Both needed for LSTM to work properly!

---

### Q7: Why is attention needed if we already have encoder state?

**A:** Encoder state is compressed. Attention provides access to details.

**Example:**
```
Encoder state: Single vector [c₀, c₁, ..., c₂₅₅]
Problem: Only 256 numbers to summarize entire English sentence!
Lost information about which words are important where.

With attention:
- Can look at: output from processing "cat", "sat", "mat" separately
- Can focus on: details relevant to Spanish word being predicted
- Can remember: all intermediate representations

Real example:
Translating "The cat sat on the mat" to Spanish
When predicting Spanish word for "cat":
- Without attention: uses only compressed summary
- With attention: can look at full encoder output
  particularly the part about "cat"
- Result: better translation!
```

---

### Q8: Why concatenate decoder output with attention output?

**A:** Combine different types of information.

**Example:**
```
Decoder output: [0.2, 0.5, 0.1, ...]  (256D)
Meaning: "I'm generating a Spanish word"

Attention output: [0.3, -0.1, 0.4, ...]  (256D)
Meaning: "Focus on these parts of English"

Concatenated: [0.2, 0.5, 0.1, ..., 0.3, -0.1, 0.4, ...]  (512D)
Meaning: "Generate Spanish word (decoder info) 
          while considering English context (attention info)"

Result: Better predictions because using both pieces of information
```

---

### Q9: Why use softmax activation in final Dense layer?

**A:** Converts outputs to probabilities (required for categorical_crossentropy).

**Example:**
```
Dense layer output: [2.1, 0.5, -0.3]  (raw scores)

With softmax:
1. Exponentiate: [e^2.1, e^0.5, e^-0.3] = [8.17, 1.65, 0.74]
2. Normalize: divide by sum (10.56)
3. Result: [0.77, 0.16, 0.07]

Interpretation:
- 77% probability of word 1
- 16% probability of word 2
- 7% probability of word 3
(All sum to 100%)

This is needed for:
- Loss calculation: -log(probability of correct word)
- Model confidence: can see how sure model is
```

---

### Q10: What does categorical_crossentropy loss mean?

**A:** Measures how wrong the model is at predicting correct word.

**Formula:**
```
loss = -log(predicted_probability_of_correct_word)

Example:
Correct word: "Hola" (index 2)
Model predicts: [0.1, 0.8, 0.05, 0.05]
Probability of correct word: 0.8 (at index 2)

Loss = -log(0.8) = 0.223

If model was better (0.95):
Loss = -log(0.95) = 0.051  (lower loss!)

If model was worse (0.5):
Loss = -log(0.5) = 0.693  (higher loss!)

Key insight:
- Correct prediction → log of large number → small loss → GOOD
- Wrong prediction → log of small number → large loss → BAD
```

---

### Q11: What does batch_size=16 mean?

**A:** Process 16 samples together before updating weights.

**Example:**
```
Total samples: 5
Batch size: 16

Since 5 < 16, all samples in one batch

With larger dataset (1000 samples, batch_size=16):
Batch 1: samples 1-16
Batch 2: samples 17-32
...
Batch 63: samples 993-1000

After each batch:
1. Calculate loss on 16 samples
2. Update weights based on those 16

Why batch?
- Faster training (process multiple at once)
- More stable updates (average over batch)
- Fits in GPU memory (16 at a time vs all)
```

---

### Q12: What's the difference between epochs and batches?

**A:** Epoch = full pass through data, Batch = portion of data.

**Example:**
```
5 samples, batch_size=2

Epoch 1:
├─ Batch 1: samples 1-2 → update weights
├─ Batch 2: samples 3-4 → update weights
└─ Batch 3: sample 5 → update weights
(completed 1 epoch)

Epoch 2: (same as epoch 1)
├─ Batch 1: samples 1-2 → update weights
├─ Batch 2: samples 3-4 → update weights
└─ Batch 3: sample 5 → update weights

Total updates per epoch: 3 (number of batches)
With epochs=100: total updates = 300
```

---

### Q13: Why does loss decrease?

**A:** Model weights are adjusted to predict better.

**Example:**
```
Initially (random weights):
Input: "Hola"
Prediction: [0.2, 0.3, 0.5]  (predicts word 3 with 50%)
Correct: word 1
Loss: -log(0.2) = 1.609  (HIGH, bad prediction)

After training (updated weights):
Input: "Hola"
Prediction: [0.7, 0.2, 0.1]  (predicts word 1 with 70%)
Correct: word 1
Loss: -log(0.7) = 0.357  (LOWER, better prediction)

Process:
1. Make prediction with current weights
2. Calculate loss
3. Calculate gradients (which direction to adjust?)
4. Move weights in direction of lower loss
5. Repeat

Over time: weights improve → predictions improve → loss decreases
```

---

### Q14: What does Adam optimizer do?

**A:** Automatically adjusts learning rate for efficient weight updates.

**Key features:**
```
Adaptive Learning Rate:
- High gradient → larger step
- Low gradient → smaller step
- Different rate for each weight

Momentum:
- Remembers past gradients
- Helps overcome flat regions
- Accelerates convergence

Comparison:
Basic SGD: All weights updated with same learning rate
Adam: Each weight has its own adaptive learning rate

Result: Adam usually converges faster!
```

---

### Q15: How do I know if model is overfitting?

**A:** Training loss decreases but validation loss increases.

**Signs of overfitting:**
```
Normal training:
Epoch 1: train_loss=5.2, val_loss=5.1
Epoch 50: train_loss=0.3, val_loss=0.4
(Both decrease together)

Overfitting:
Epoch 1: train_loss=5.2, val_loss=5.1
Epoch 50: train_loss=0.1, val_loss=1.2
(Train decreases, validation increases!)

Why?
- Model memorizes training data
- Doesn't generalize to new data

Solutions:
1. Use Dropout layer (randomly removes neurons)
2. Reduce model size
3. Use more training data
4. Use regularization
5. Early stopping (stop before overfitting)
```

---

## Common Parameter Adjustments

### If Model is Not Learning Fast:

```python
# Increase learning rate (in optimizer)
model.compile(optimizer=Adam(learning_rate=0.01), ...)
# Default: 0.001, try: 0.01

# Or increase model capacity
Dense(512)  # instead of Dense(256)
LSTM(512)   # instead of LSTM(256)

# Or increase training time
epochs=200  # instead of epochs=100
```

### If Model is Overfitting:

```python
# Add Dropout
from keras.layers import Dropout
Dense(256)(x)
Dropout(0.5)(x)  # Drop 50% of neurons

# Reduce model size
Dense(128)  # instead of Dense(256)
LSTM(128)   # instead of LSTM(256)

# Add regularization
Dense(256, kernel_regularizer='l2')(x)
```

### If Training is Unstable (Loss jumps around):

```python
# Reduce learning rate
optimizer=Adam(learning_rate=0.0001)

# Reduce batch size to see more updates
batch_size=8  # instead of batch_size=16

# Add gradient clipping
from keras.optimizers import Adam
optimizer=Adam(clipvalue=0.5)
```

---

## Quick Debugging Checklist

- [ ] Check input/output shapes match expected dimensions
- [ ] Verify tokenizer vocabulary is reasonable size
- [ ] Check that padding length is sufficient
- [ ] Ensure batch_size is less than total samples
- [ ] Check loss is decreasing over epochs
- [ ] Verify encoder and decoder hidden dimensions match
- [ ] Check that attention receives correct inputs (Q, K, V)
- [ ] Confirm one-hot encoding is correct
- [ ] Test on small dataset first (overfit easily → works!)
- [ ] Print sample predictions to verify reasonableness

---

## Key Formulas Cheat Sheet

```
TOKENIZATION:
word_to_index: "hello" → 1

EMBEDDING:
embedding: index → vector
1 → [0.2, -0.1, 0.5, ..., 0.3]  (256D)

ATTENTION:
Q = X @ W_q
K = X @ W_k
V = X @ W_v
scores = Q @ K^T / √d_k
weights = softmax(scores)
output = weights @ V

CLASSIFICATION:
logits = Dense(vocab_size)(input)
probabilities = softmax(logits)
loss = -log(probability of correct class)

GRADIENT DESCENT:
new_weight = old_weight - learning_rate × gradient
```

---

## Model Architecture Summary

```
Input English: [batch, seq_length]
    ↓
Embedding: [batch, seq_length, 256]
    ↓
LSTM Encoder: outputs [batch, seq_length, 256]
             states [batch, 256]
    ↓
Input Spanish: [batch, seq_length-1]
    ↓
Embedding: [batch, seq_length-1, 256]
    ↓
LSTM Decoder (init with encoder states): [batch, seq_length-1, 256]
    ↓
Attention (Q from decoder, K,V from encoder): [batch, seq_length-1, 256]
    ↓
Concatenate: [batch, seq_length-1, 512]
    ↓
Dense + Softmax: [batch, seq_length-1, vocab_size]
    ↓
Output Spanish Probabilities

Loss = categorical_crossentropy(one_hot(target), output)
```

---

## When to Use What

| Component | When to Use | When NOT to Use |
|-----------|------------|-----------------|
| Embedding | Text input | Numerical input |
| LSTM | Sequential data | Image data |
| Attention | Long sequences | Short sequences (might hurt) |
| Dropout | To prevent overfitting | When underfitting |
| Dense | Final predictions | Intermediate layers (usually) |
| Softmax | Multi-class classification | Regression problems |
| Adam | Most problems | Sparse high-dimensional data (try Adagrad) |
| Batch size 16 | Default, works for most | Increase for GPU, decrease for memory issues |

---

## Understanding Loss Values

```
Loss = 5.0  → Very bad (random guessing)
Loss = 1.0  → Not great
Loss = 0.5  → Okay
Loss = 0.1  → Good
Loss = 0.01 → Very good

But remember: absolute value depends on:
- Vocabulary size
- Sequence length
- Data difficulty
```

---

## Tips for Success

1. **Start small**: Test on tiny dataset first
2. **Overfit intentionally**: Train on 1 sample → verify it works
3. **Check shapes**: Print tensor shapes at each step
4. **Visualize training**: Always plot loss curve
5. **Test incrementally**: Add features one by one
6. **Use validation**: Split data into train/val/test
7. **Save best model**: Save weights when validation loss is lowest
8. **Experiment**: Try different hyperparameters

---

## Resources for Deeper Learning

1. **Self-Attention**: "Attention is All You Need" paper
2. **Transformers**: HuggingFace transformers library
3. **Seq2Seq**: Official TensorFlow tutorials
4. **LSTM**: Christopher Olah's blog posts
5. **Keras**: Official Keras documentation

---

This guide covers the most important concepts and common questions!
For detailed explanations, see the other documents:
- Transformers_Keras_Complete_Guide.md (full explanations)
- Visual_Reference_Guide.md (diagrams and visualizations)
- Fully_Commented_Code.md (code with comments)


# 2. Transformers with Keras - Complete Guide

## Overview of the Notebook

This notebook teaches you how to build a **Transformer model for machine translation** (English to Spanish) using Keras. It uses a **Sequence-to-Sequence (Seq2Seq) architecture with Self-Attention**, which is the core technology behind modern AI models like ChatGPT.

---

## **Part 1: Easy Concepts Explanation**

## What is a Transformer?

Imagine you're translating a sentence from English to Spanish. A transformer is like a smart assistant that:
1. **Reads the entire English sentence** first (not word by word)
2. **Understands which words are important** for each word in the Spanish translation
3. **Generates the Spanish sentence** based on this understanding

## Key Components

### 1. **Embedding Layer**
- **What it does:** Converts words into numbers
- **Why needed:** Neural networks understand numbers, not words
- **Example:** 
  - "Hello" → [0.2, 0.5, -0.3] (a list of decimal numbers)
  - "World" → [0.1, 0.3, 0.8]

### 2. **LSTM (Long Short-Term Memory)**
- **What it does:** Processes sequences of words and remembers important information
- **Think of it as:** A smart memory that reads words one by one and remembers what's important
- **Example:** When reading "I love pizza", it remembers that "I" is the subject even when processing later words

### 3. **Self-Attention Mechanism** ⭐ (The Star of Transformers)
- **What it does:** Makes the model focus on relevant words when translating each word
- **Think of it as:** When translating "The cat sat on the mat", when processing "cat", the model pays attention to "the" (related article) and "sat" (related verb)
- **How it works:**
  - **Query (Q):** "What word am I looking for?"
  - **Key (K):** "What information do I have?"
  - **Value (V):** "The actual word/meaning"

### 4. **Encoder-Decoder Architecture**
- **Encoder:** Reads the English sentence and creates a summary (called "context vector")
- **Decoder:** Uses this summary to generate the Spanish translation, word by word

---

## **Part 2: Step-by-Step Code Explanation**

## Step 0: Import Libraries

```python
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
```
**What it does:** Suppresses warning messages from TensorFlow (makes output cleaner)

```python
import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
from keras.layers import Layer
import warnings
warnings.simplefilter('ignore', FutureWarning)
```

**Explanation:**
- `numpy`: For working with arrays/numbers
- `keras.models.Model`: Base class to build neural network models
- `Input, LSTM, Dense, Embedding, Dropout`: Building blocks for neural networks
  - `Input`: Defines input layer
  - `LSTM`: Processes sequences
  - `Dense`: Regular neural network layer
  - `Embedding`: Converts words to numbers
  - `Dropout`: Prevents overfitting
- `Tokenizer`: Converts text to numbers
- `pad_sequences`: Makes all sentences same length
- `backend as K`: For doing mathematical operations

---

## Step 1: Data Preparation

### 1.1 Define Training Data

```python
input_texts = [
    "Hello.", "How are you?", "I am learning machine translation.", 
    "What is your name?", "I love programming."
]
target_texts = [
    "Hola.", "¿Cómo estás?", "Estoy aprendiendo traducción automática.", 
    "¿Cuál es tu nombre?", "Me encanta programar."
]
```

**What it does:** Creates English-Spanish sentence pairs for training

```python
target_texts = ["startseq " + x + " endseq" for x in target_texts]
```

**What it does:** Adds special markers to target sentences:
- `startseq`: Tells decoder "start generating translation"
- `endseq`: Tells decoder "stop, translation is done"
- **Example:** "Hola." becomes "startseq Hola. endseq"

### 1.2 Tokenization (Converting Words to Numbers)

```python
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)
```

**Step-by-step breakdown:**

1. Create a Tokenizer object
2. `fit_on_texts()`: Learn all unique words in English sentences
   - **Example:** learns words = {"hello", "how", "are", "you", ...}
   - Creates word-to-number mapping:
     - "hello" → 1
     - "how" → 2
     - "are" → 3
3. `texts_to_sequences()`: Convert sentences to numbers
   - **Example:** "Hello how" → [1, 2]

```python
output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1
```

**Explanation:**
- Same process for Spanish sentences
- `vocab_size`: Total number of unique words + 1
- The "+1" is for unknown words (words not in training data)

### 1.3 Padding Sequences (Making All Sentences Same Length)

```python
max_input_length = max([len(seq) for seq in input_sequences])
max_output_length = max([len(seq) for seq in output_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')
```

**What it does:** Makes all sentences the same length by adding zeros

**Example:**
- Before: [1, 2, 3], [1, 2]
- After: [1, 2, 3], [1, 2, 0] (add 0 to match length 3)
- `padding='post'` means add zeros at the end

**Why needed:** Neural networks like consistent input sizes

### 1.4 Prepare Decoder Input and Output

```python
decoder_input_data = output_sequences[:, :-1]  # All but last element
decoder_output_data = output_sequences[:, 1:]   # All but first element
```

**What it does:** Shifts the sequence for training

**Example:** If output_sequences = [1, 2, 3, 4]
- `decoder_input_data` = [1, 2, 3] (feed these to decoder)
- `decoder_output_data` = [2, 3, 4] (predict these)

**Why:** The decoder learns to predict the next word

```python
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])
```

**What it does:** Converts numbers to one-hot encoding

**Example:** If vocab_size = 5 and word = 2:
- Number: [2]
- One-hot: [0, 0, 1, 0, 0] (1 at position 2, 0s everywhere else)

**Why:** Required for `categorical_crossentropy` loss function

---

## Step 2: Self-Attention Layer (Core of Transformers)

### Understanding Self-Attention

**Analogy:** When reading "The cat sat on the mat", when you process "cat":
- You pay 80% attention to "the" (article modifying cat)
- You pay 15% attention to "sat" (verb for cat's action)
- You pay 5% attention to other words

The model learns these attention percentages!

### The SelfAttention Class

```python
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
```

**What it does:** Creates a new layer type (inherits from Keras Layer)

### Build Method (Initialize Weights)

```python
def build(self, input_shape):
    feature_dim = input_shape[0][-1]
    
    self.Wq = self.add_weight(
        shape=(feature_dim, feature_dim),
        initializer='glorot_uniform',
        trainable=True,
        name='Wq'
    )
```

**What it does:** Creates trainable weight matrices

**Components:**
- `feature_dim`: Size of the vector representation (usually 256 in our model)
- `self.Wq`: Weight matrix for Query (shape: 256×256)
- `self.Wk`: Weight matrix for Key (shape: 256×256)
- `self.Wv`: Weight matrix for Value (shape: 256×256)
- `initializer='glorot_uniform'`: Random starting values (good for neural networks)
- `trainable=True`: These weights will be updated during training

### Call Method (The Actual Computation)

```python
def call(self, inputs):
    q, k, v = inputs
    
    q = K.dot(q, self.Wq)  # Query = input × Wq
    k = K.dot(k, self.Wk)  # Key = input × Wk
    v = K.dot(v, self.Wv)  # Value = input × Wv
```

**What it does:** Transforms inputs into Query, Key, and Value

**Example:**
- Input shape: (batch_size=2, seq_length=5, features=256)
- Query shape: (2, 5, 256) - 5 words, each represented by 256 numbers

```python
scores = K.batch_dot(q, k, axes=[2, 2])
dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
scores = scores / K.sqrt(dk)
```

**What it does:** Computes attention scores using dot product

**Step-by-step:**
1. `K.batch_dot(q, k)`: Multiply queries with keys
   - Creates a score matrix showing similarity between words
   - Shape: (batch_size, seq_len, seq_len) = (2, 5, 5)
   - Element [i,j] = how much word i attends to word j

2. `scores / sqrt(dk)`: Scale the scores
   - `dk` = feature dimension (256)
   - `sqrt(256)` = 16
   - Why scale? Prevents scores from becoming too large (which breaks softmax)

```python
attention_weights = K.softmax(scores, axis=-1)
output = K.batch_dot(attention_weights, v)
```

**What it does:** Convert scores to weights and apply to values

**Step-by-step:**
1. `K.softmax()`: Convert attention scores to weights (0 to 1, sum to 1)
   - Example: [2.0, 1.0, 0.5] → [0.66, 0.24, 0.10]
   - These weights show "importance" of each word

2. `K.batch_dot(attention_weights, v)`: Apply weights to values
   - Creates output = weighted combination of all words
   - High attention weight = that word contributes more to output

---

## Step 3: Model Architecture

### Building the Encoder

```python
encoder_inputs = Input(shape=(max_input_length,))
encoder_embedding = Embedding(input_vocab_size, 256)(encoder_inputs)
```

**What it does:**
- `Input()`: Define input layer (expects sentence as numbers)
- `Embedding()`: Convert word numbers to 256-dimensional vectors
  - Input: [1, 2, 3, 0] (4 words)
  - Output: [[0.1, 0.2, ...], [0.3, 0.4, ...], ...] (4 vectors of 256 numbers each)

```python
encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]
```

**What it does:** Processes the English sentence with LSTM

**Components:**
- `LSTM(256)`: LSTM with 256 hidden units (memory cells)
- `return_sequences=True`: Return output for each time step (each word)
- `return_state=True`: Also return final hidden state and cell state
- **Outputs:**
  - `encoder_outputs`: All hidden states (one per word)
  - `state_h`: Final hidden state (256 numbers - context summary)
  - `state_c`: Final cell state (256 numbers - long-term memory)

**Why these three outputs?**
- The encoder reads all 5 English words
- `encoder_outputs`: Detailed information about each word
- `state_h`, `state_c`: Compressed summary to pass to decoder

### Building the Decoder

```python
decoder_inputs = Input(shape=(max_output_length - 1,))
decoder_embedding = Embedding(output_vocab_size, 256)(decoder_inputs)
```

**What it does:** Same as encoder - converts Spanish word numbers to vectors

```python
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
```

**What it does:** Processes Spanish with LSTM, starting from encoder's final state

**Key difference:**
- `initial_state=encoder_states`: Start decoder with encoder's learned context
- This is how the decoder "knows about" the English sentence!

### Adding Attention

```python
self_attention = SelfAttention()
attention_output = self_attention([decoder_outputs, encoder_outputs, encoder_outputs])
```

**What it does:** Decoder pays attention to encoder outputs

**Parameters:**
- Query (decoder_outputs): "What do I need to translate?"
- Key (encoder_outputs): "What information is in English?"
- Value (encoder_outputs): "What English words should I look at?"

**Result:** Attention highlights which English words are important for each Spanish word

### Combining Outputs

```python
decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_output])
decoder_dense = Dense(output_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)
```

**What it does:**
1. Concatenate decoder LSTM output with attention output
   - Before: (batch, seq_len, 256)
   - After: (batch, seq_len, 512) - doubled in size
   
2. Pass through Dense layer
   - Input: 512 dimensions
   - Output: vocab_size dimensions (one score per possible word)
   
3. Apply softmax
   - Converts scores to probabilities (0-1, sum to 1)
   - Example: [2.1, 0.5, 0.3] → [0.8, 0.15, 0.05]

### Creating the Full Model

```python
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
```

**What it does:**
- Links all components together
- **Inputs:** English sentences + shifted Spanish sentences
- **Output:** Predicted next Spanish word
- **Optimizer:** Adam (adaptive learning rate optimizer)
- **Loss:** Categorical crossentropy (measures how wrong predictions are)
- **Metrics:** Accuracy (percentage of correct predictions)

---

## Step 4: Training the Model

```python
history = model.fit(
    [input_sequences, decoder_input_data], 
    decoder_output_data, 
    epochs=100, 
    batch_size=16
)
```

**What it does:** Train the model

**Parameters:**
- `[input_sequences, decoder_input_data]`: Inputs (English + shifted Spanish)
- `decoder_output_data`: Expected outputs (actual Spanish words)
- `epochs=100`: Run through all data 100 times
- `batch_size=16`: Process 16 samples at a time

**What happens during training:**
1. Model makes predictions
2. Compares predictions to actual Spanish words
3. Calculates error (loss)
4. Updates weights to reduce error
5. Repeats for all data and all epochs

---

## Step 5: Visualizing Results

```python
import matplotlib.pyplot as plt
plt.plot(history.history['loss'])
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()
```

**What it shows:**
- Loss should decrease over time (model getting better)
- If loss increases, model might be overfitting

---

## **Part 3: How Everything Works Together**

## Complete Translation Flow

### 1. **Input Processing**
```
English: "Hello how are you"
         ↓
Tokenize: [1, 2, 3, 4]
         ↓
Embed: [[0.1, 0.2, ...], [0.3, 0.4, ...], ...]
```

### 2. **Encoder Processing**
```
Embedded input → LSTM → Reads entire English sentence
                 ↓
            Encoder state (256 numbers) = "Summary of English"
```

### 3. **Decoder + Attention**
```
Spanish input (startseq) → Embed → Decoder LSTM
                                    ↓
                        Uses encoder state to start
                                    ↓
                        Attention layer looks at:
                        - What decoder needs (Query)
                        - What encoder knows (Key, Value)
                                    ↓
                        Output = relevant parts of English
```

### 4. **Prediction**
```
Decoder output + Attention → Concatenate → Dense layer → Softmax
                                                           ↓
                                              Probability for each word
                                                           ↓
                                           Highest probability = Predicted word
```

### 5. **Repeat**
```
Predicted word → Feed back as input for next step
                ↓
            Predict next word
                ↓
            Continue until "endseq" or max length
```

---

## **Practice Exercises Explanation**

## Exercise 1: Comparing Initializers

**What it does:** Compare `glorot_uniform` vs `he_uniform` weight initialization

**Explanation:**
- Different initializers start weights differently
- This affects how fast the model learns
- The exercise shows which initializer works better

**Key change:**
```python
# Instead of 'glorot_uniform'
initializer='he_uniform'
```

**Result:** A graph comparing training loss between both initializers

## Exercise 2: Comparing Optimizers

**What it does:** Compare Adam vs Adagrad optimizers

**Explanation:**
- Different optimizers update weights differently
- Adam usually faster but Adagrad sometimes better
- Shows which optimizer works better for this problem

**Key change:**
```python
# Instead of 'adam'
model.compile(optimizer='adagrad', loss='categorical_crossentropy')
```

---

## **Summary Table**

| Component | Purpose | Simple Analogy |
|-----------|---------|-----------------|
| **Embedding** | Convert words to numbers | Dictionary that assigns meaning to words |
| **LSTM** | Process sequences | Smart assistant that remembers context |
| **Self-Attention** | Focus on important words | Highlighting relevant parts of text |
| **Encoder** | Understand input | Reading and understanding English |
| **Decoder** | Generate output | Speaking in Spanish based on understanding |
| **Dense Layer** | Predict next word | Guessing which word comes next |

---

## **Key Takeaways**

1. **Transformers use Self-Attention** to understand relationships between words
2. **Encoder-Decoder** splits the task: understand input, generate output
3. **Attention mechanism** allows decoder to focus on important parts
4. **Training** adjusts weights to minimize prediction errors
5. **Hyperparameters** (epochs, batch_size, initializer, optimizer) affect learning

This architecture forms the basis of ChatGPT, BERT, and other modern AI models!


# 3. Visual Reference Guide - Transformers with Keras



## 1. Self-Attention Mechanism - Visual Breakdown

### Step 1: Create Query, Key, Value

```
Input Embeddings:    "The cat sat"
┌─────┬─────┬─────┐
│The  │ cat │ sat │  256 dimensions each
└─────┴─────┴─────┘
     ↓↓↓
Multiply by Weight Matrices (Wq, Wk, Wv)
     ↓↓↓
Query (Q):           "What I'm looking for"
Key (K):             "What each word represents"
Value (V):           "Actual word content"
```

### Step 2: Calculate Attention Scores

```
Query × Key^T = Attention Scores

         Key
    The  cat  sat
Qu ┌──────────────┐
e  │ 2.1  0.5  1.2│ The (attention to The, cat, sat)
   │ 0.3  3.0  0.8│ cat (attention to The, cat, sat)
   │ 1.2  0.4  2.5│ sat (attention to The, cat, sat)
   └──────────────┘

Higher number = pay more attention
```

### Step 3: Normalize with Softmax

```
Raw Scores:  [2.1, 0.5, 1.2]
                ↓
Divide by √256 = √dk
                ↓
Apply Softmax:
                ↓
Attention Weights: [0.65, 0.12, 0.23]
(Notice: sum = 1.0, all between 0-1)
```

### Step 4: Apply to Values

```
Attention [0.65, 0.12, 0.23] × Values
    ↓
Output = 0.65×(The vector) + 0.12×(cat vector) + 0.23×(sat vector)
    ↓
Result: Context-aware representation of "The"
```

---

## 2. Full Architecture Diagram

```
═══════════════════════════════════════════════════════════════════

ENGLISH INPUT: "Hello how are you"
        ↓
    Tokenize: [1, 2, 3, 4]
        ↓
    Embedding (256D): 4×256 matrix
        ↓
    ╔═════════════════════════════════════════╗
    ║         ENCODER (LSTM)                  ║
    ║  Reads entire English sentence          ║
    ║  Returns: outputs + hidden state        ║
    ╚═════════════════════════════════════════╝
        ↓
    Encoder Outputs: [H₁, H₂, H₃, H₄] (context for each word)
    Encoder State: C (context summary, 256D)
        ↓↓↓
        ║
        ║ (Passed to Decoder)
        ║
        ↓
    SPANISH INPUT (shifted): "startseq Hola"
        ↓
    Tokenize: [1, 2]
        ↓
    Embedding: 2×256 matrix
        ↓
    ╔═════════════════════════════════════════╗
    ║       DECODER (LSTM)                    ║
    ║  Initial state = Encoder state C        ║
    ║  Uses English context to translate      ║
    ╚═════════════════════════════════════════╝
        ↓
    Decoder Outputs: [D₁, D₂]
        ↓
    ╔═════════════════════════════════════════╗
    ║    SELF-ATTENTION                       ║
    ║  Q = Decoder outputs                    ║
    ║  K, V = Encoder outputs                 ║
    ║  Focus on relevant English words        ║
    ╚═════════════════════════════════════════╝
        ↓
    Attention Outputs: [A₁, A₂]
        ↓
    Concatenate: [D₁, A₁] + [D₂, A₂] (512D each)
        ↓
    ╔═════════════════════════════════════════╗
    ║  Dense Layer (output_vocab_size)        ║
    ║  Softmax activation                     ║
    ╚═════════════════════════════════════════╝
        ↓
    Output Probabilities:
    Position 1: [0.05, 0.8, 0.1, 0.05] (Hola = 0.8)
    Position 2: [0.1, 0.05, 0.7, 0.15] (está = 0.7)
        ↓
    PREDICTED SPANISH: "Hola está"

═══════════════════════════════════════════════════════════════════
```

---

## 3. Data Flow - Detailed Dimensions

```
ENCODER:
────────────────────────────────────────
Input:  (batch=1, seq=5) 
        [1, 2, 3, 4, 0] (English sentence + padding)
        ↓
Embedding: (batch=1, seq=5, features=256)
        [[0.1, 0.2, ..., 0.5],     ← "Hello" as 256 numbers
         [0.3, 0.4, ..., 0.6],     ← "How" as 256 numbers
         [0.5, 0.6, ..., 0.7],     ← "are" as 256 numbers
         [0.2, 0.3, ..., 0.4],     ← "you" as 256 numbers
         [0.0, 0.0, ..., 0.0]]     ← padding as zeros
        ↓
LSTM Forward: (batch=1, seq=5, hidden=256)
        [[h₁], [h₂], [h₃], [h₄], [h₅]]  ← hidden state per word
        ↓
Final State: (batch=1, hidden=256)
        [c₀, c₁, ..., c₂₅₅]  ← compressed summary


DECODER:
────────────────────────────────────────
Input:  (batch=1, seq=4)
        [1, 2, 3, 0] (Spanish + padding, shifted)
        ↓
Embedding: (batch=1, seq=4, features=256)
        ↓
LSTM Forward (starting with encoder state): (batch=1, seq=4, hidden=256)
        [[d₁], [d₂], [d₃], [d₄]]
        ↓
ATTENTION:
Query:  (batch=1, seq=4, dim=256) from Decoder
Key:    (batch=1, seq=5, dim=256) from Encoder
Value:  (batch=1, seq=5, dim=256) from Encoder
        ↓
Attention Scores: (batch=1, seq=4, seq=5)
        [[score 1→1, score 1→2, score 1→3, score 1→4, score 1→5],
         [score 2→1, score 2→2, score 2→3, score 2→4, score 2→5],
         ...
        ]
        ↓
Attention Output: (batch=1, seq=4, dim=256)
        [[context for word 1],
         [context for word 2],
         ...
        ]
        ↓
Concatenate: (batch=1, seq=4, dim=512)
        [[d₁ + attention₁],
         [d₂ + attention₂],
         ...
        ]
        ↓
Dense Layer: (batch=1, seq=4, vocab_size=15)
        [[0.05, 0.8, 0.1, ...],   ← probabilities for word 1
         [0.1, 0.05, 0.7, ...],   ← probabilities for word 2
         ...
        ]
```

---

## 4. Code Flow Summary

### Data Preparation Phase

```
1. Input & Target Texts
   ↓
2. Add startseq/endseq
   ↓
3. Tokenization (word → number)
   Input Tokenizer: {"hello": 1, "how": 2, ...}
   Output Tokenizer: {"hola": 1, "cómo": 2, ...}
   ↓
4. Convert to sequences
   "Hello" → [1]
   "How are you" → [2, 3, 4]
   ↓
5. Padding (all same length)
   [1] → [1, 0, 0, 0, 0]
   [2, 3, 4] → [2, 3, 4, 0, 0]
   ↓
6. Prepare decoder data
   Shift Spanish: [1, 2, 3] input, [2, 3, 4] output
   ↓
7. One-hot encode
   2 → [0, 0, 1, 0, ...]
```

### Model Building Phase

```
1. Define inputs
   encoder_inputs = Input(shape=(max_input_length,))
   decoder_inputs = Input(shape=(max_output_length-1,))
   ↓
2. Create encoder
   embedding → LSTM → (outputs, state_h, state_c)
   ↓
3. Create decoder
   embedding → LSTM(initial_state=encoder states) → decoder outputs
   ↓
4. Add attention
   SelfAttention([decoder_outputs, encoder_outputs, encoder_outputs])
   ↓
5. Combine & predict
   Concatenate → Dense → Softmax
   ↓
6. Link components
   Model([encoder_inputs, decoder_inputs], final_output)
   ↓
7. Compile
   optimizer='adam', loss='categorical_crossentropy'
```

### Training Phase

```
FOR each epoch (1 to 100):
   FOR each batch of 16 samples:
      1. Forward pass (make predictions)
      2. Calculate loss (how wrong are we?)
      3. Backward pass (calculate gradients)
      4. Update weights (move in direction of less loss)
   END
   Print loss for this epoch
END
```

---

## 5. Self-Attention in Detail with Numbers

### Example: Processing "cat" in "The cat sat"

```
Step 1: Initial Embeddings
────────────────────────────
The:  [0.2, -0.1, 0.5, ..., 0.3]     (256 dimensions)
cat:  [0.4, 0.3, -0.2, ..., 0.1]     (256 dimensions)
sat:  [0.1, 0.2, 0.6, ..., -0.1]     (256 dimensions)


Step 2: Create Q, K, V for "cat"
────────────────────────────────
Q_cat = cat_embedding × W_q = [0.1, 0.2, -0.1, ..., 0.4]
K_the = the_embedding × W_k  = [0.3, -0.2, 0.1, ..., 0.5]
K_cat = cat_embedding × W_k  = [0.5, 0.1, 0.2, ..., -0.1]
K_sat = sat_embedding × W_k  = [0.2, 0.4, -0.3, ..., 0.2]

V_the = [0.15, -0.05, 0.25, ..., 0.15]
V_cat = [0.20, 0.15, -0.10, ..., 0.05]
V_sat = [0.05, 0.10, 0.30, ..., -0.05]


Step 3: Calculate Scores (Q_cat · K)
─────────────────────────────────────
score(cat, the) = Q_cat · K_the = 0.1×0.3 + 0.2×(-0.2) + ... = 2.1
score(cat, cat) = Q_cat · K_cat = 0.1×0.5 + 0.2×0.1 + ... = 3.2
score(cat, sat) = Q_cat · K_sat = 0.1×0.2 + 0.2×0.4 + ... = 1.8

Scores: [2.1, 3.2, 1.8]


Step 4: Scale by √d_k (d_k = 256, √256 = 16)
──────────────────────────────────────────────
Scaled: [2.1/16, 3.2/16, 1.8/16] = [0.13, 0.20, 0.11]


Step 5: Apply Softmax
──────────────────────
e^0.13 = 1.14
e^0.20 = 1.22
e^0.11 = 1.12

Sum = 1.14 + 1.22 + 1.12 = 3.48

Weights: [1.14/3.48, 1.22/3.48, 1.12/3.48] = [0.33, 0.35, 0.32]


Step 6: Apply to Values
──────────────────────
Output = 0.33×V_the + 0.35×V_cat + 0.32×V_sat
       = 0.33×[0.15, -0.05, 0.25, ...] 
         + 0.35×[0.20, 0.15, -0.10, ...] 
         + 0.32×[0.05, 0.10, 0.30, ...]
       = [0.049, 0.036, 0.098, ...]  (256 dimensions)

This output represents "cat" in context of full sentence!
```

### Interpretation

```
Attention weights for "cat": [0.33, 0.35, 0.32]
                              the   cat   sat

Meaning:
- 33% attention to "the" (article modifying cat)
- 35% attention to "cat" (itself - it's important!)
- 32% attention to "sat" (verb describing cat's action)

The model learns that all three are important for understanding "cat"
in this context!
```

---

## 6. Loss Visualization Explanation

```
EPOCH 1:  Loss = 5.2  ██████████████████ High loss (bad predictions)
EPOCH 10: Loss = 3.8  ████████████       Getting better
EPOCH 20: Loss = 2.1  ███████             Improving
EPOCH 50: Loss = 0.8  ███                 Good predictions
EPOCH 100:Loss = 0.3  █                   Very good predictions

Graph appearance:
Loss ↑
  5 |████                                  ← Starting high
    |  ████                                ← Steep drop (learning)
    |      ███                             ← Still improving
    |         ███                          ← Slowing down
  1 |            ██                        ← Converging
    |              ██                      ← Plateau (model saturated)
    |________________→ Epochs
```

**Good signs:**
- Consistent decrease in loss
- Loss approaches 0
- No sudden spikes upward

**Bad signs:**
- Loss increases suddenly (overfitting)
- Loss stays flat (not learning)
- Very noisy (unstable training)

---

## 7. Key Equations Reference

### Attention Mechanism
```
Q = Input × W_q          (Query transformation)
K = Input × W_k          (Key transformation)
V = Input × W_v          (Value transformation)

Scores = Q × K^T         (Similarity between queries and keys)
Scores = Scores / √d_k   (Scaling)

AttentionWeights = softmax(Scores)    (Normalize to probabilities)
Output = AttentionWeights × V         (Apply weights to values)
```

### One-hot Encoding
```
Word index = 2, Vocab size = 5

One-hot = [0, 0, 1, 0, 0]
           0  1  2  3  4  ← positions
                ↑ This is 1
```

### Softmax
```
softmax(x_i) = e^x_i / Σ(e^x_j)

Example:
x = [2.0, 1.0, 0.5]
e^x = [7.4, 2.7, 1.6]
softmax = [7.4/11.7, 2.7/11.7, 1.6/11.7] = [0.63, 0.23, 0.14]
```

---

## 8. Training Iteration Example

```
ITERATION 1:
───────────

Input:
  English: "Hello" → Tokens: [1] → Padded: [1, 0, 0, 0, 0]
  Spanish: "startseq" → Tokens: [1] → Padded: [1, 0, 0, 0]

Forward Pass:
  Encoder processes [1, 0, 0, 0, 0]
  ↓
  Decoder processes [1, 0, 0, 0] with encoder state
  ↓
  Attention focuses on relevant parts
  ↓
  Predictions: [[0.05, 0.8, 0.1, 0.05, ...]]
  Predicted word: "Hola" (index 2, prob 0.8)

Actual Output:
  Word at position 1: "Hola" (one-hot: [0, 0, 1, 0, ...])

Loss Calculation:
  -ln(0.8) = 0.22  ← Prediction was good!

Backpropagation:
  Calculate gradient for all weights
  ↓
Weight Update:
  new_weight = old_weight - learning_rate × gradient
  Example: W = 0.5 - 0.001 × (-2.3) = 0.5 + 0.0023 = 0.5023

ITERATION 2 (next batch):
  Weight = 0.5023 (slightly updated)
  Process continues...
```

---

## 9. Comparison: Glorot vs He Initialization

```
GLOROT UNIFORM (default in our code):
─────────────────────────────────────
Range: [-√(6/(n_in + n_out)), +√(6/(n_in + n_out))]
For 256×256: [-0.035, +0.035]

Good for: Tanh and Sigmoid activations

Learning curve typically:
Loss ↓
  5 |████
    |  ████
    |      ████
  1 |           ████  ← Slower convergence
    |________________→


HE UNIFORM (for ReLU):
──────────────────────
Range: [-√(6/n_in), +√(6/n_in)]
For 256×256: [-0.050, +0.050]  (wider range)

Good for: ReLU activations

Learning curve typically:
Loss ↓
  5 |████
    |  ████████
    |      ████  ← Faster initial drop
  1 |        ██  ← But might overshoot
    |________________→


RESULT OF PRACTICE EXERCISE 1:
The plot shows which converges faster!
```

---

## 10. Optimizer Comparison

```
ADAM (Adaptive Moment Estimation):
──────────────────────────────────
Updates weights using:
- Momentum (remember past gradients)
- Adaptive learning rate (different for each weight)

Learning curve:
Loss ↓
  5 |████
    |  ████████
    |      ████████  ← Smooth, consistent
  1 |            ██  ← Fast and reliable
    |________________→


ADAGRAD (Adaptive Gradient):
────────────────────────────
Updates weights using:
- Accumulated gradient history
- Adaptive learning rate

Learning curve:
Loss ↓
  5 |████
    |  ████████
    |      ████████
  1 |           ███  ← Learning rate decreases over time
    |              ██ ← Might get stuck
    |________________→


KEY DIFFERENCES:
              Adam        Adagrad
Speed:        Fast        Slow
Stability:    Smooth      Can plateau
Memory:       More        Less
Best for:     Most tasks  Sparse data
```

---

This visual guide complements the code by showing how data flows, how attention works,
and what the model learns at each step!


# 4. Transformers with Keras - Fully Commented Code


## SECTION 1: Setup and Imports

### Suppress TensorFlow Warnings
```python
import os
# Suppress TensorFlow warning messages (mainly for CPU users)
# Line explanation: Set environment variable to reduce verbose output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
```

### Import Essential Libraries
```python
import numpy as np
# NumPy: For numerical operations and array manipulation

from keras.models import Model
# Model: Base class for building neural networks (functional API)

from keras.layers import Input, LSTM, Dense, Embedding, Dropout
# Input: Defines input layer shape
# LSTM: Long Short-Term Memory - processes sequences and remembers context
# Dense: Fully connected layer for final predictions
# Embedding: Converts integers (word tokens) to dense vector representations
# Dropout: Regularization to prevent overfitting (randomly drops neurons during training)

from tensorflow.keras.preprocessing.text import Tokenizer
# Tokenizer: Converts text sentences into sequences of integers

from keras.preprocessing.sequence import pad_sequences
# pad_sequences: Makes all sequences same length by adding zeros

from tensorflow.keras import backend as K
# backend as K: Low-level TensorFlow operations for mathematical computations

from keras.layers import Layer
# Layer: Base class for creating custom layers (used for SelfAttention)

import warnings
warnings.simplefilter('ignore', FutureWarning)
# Suppress FutureWarning messages for cleaner output
```

---

## SECTION 2: Data Preparation

### 2.1 Define Training Data (English-Spanish Pairs)

```python
# Sample parallel sentences: English phrases and their Spanish translations
input_texts = [
    "Hello.",                                      # Example 1
    "How are you?",                               # Example 2
    "I am learning machine translation.",         # Example 3
    "What is your name?",                         # Example 4
    "I love programming."                         # Example 5
]

target_texts = [
    "Hola.",                                      # Translation of "Hello"
    "¿Cómo estás?",                              # Translation of "How are you?"
    "Estoy aprendiendo traducción automática.",  # Translation of Example 3
    "¿Cuál es tu nombre?",                       # Translation of Example 4
    "Me encanta programar."                       # Translation of Example 5
]

# Add special tokens to target (Spanish) sentences
# "startseq" tells decoder: START generating translation
# "endseq" tells decoder: STOP, translation is complete
target_texts = ["startseq " + x + " endseq" for x in target_texts]

# Result example:
# "Hola." becomes "startseq Hola. endseq"
# This helps decoder learn when to start and stop translating
```

### 2.2 Tokenization (Convert Words to Numbers)

```python
# === Tokenize INPUT (English) sentences ===

# Create tokenizer object
# Tokenizer learns vocabulary by analyzing text
input_tokenizer = Tokenizer()

# fit_on_texts: Learn all unique words in English sentences
# It creates a word-to-number mapping
# Example: {"hello": 1, "how": 2, "are": 3, ...}
input_tokenizer.fit_on_texts(input_texts)

# Convert sentences to sequences of integers
# "Hello How" → [1, 2]
# Text is replaced with its corresponding number
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# Result example:
# input_texts[0] = "Hello." → input_sequences[0] = [1]
# input_texts[1] = "How are you?" → input_sequences[1] = [2, 3, 4]


# === Tokenize OUTPUT (Spanish) sentences ===

# Same process for Spanish sentences
output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)  # Learn Spanish vocabulary
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

# Result example:
# target_texts[0] = "startseq Hola. endseq" → [1, 2, 3]
# target_texts[1] = "startseq ¿Cómo estás? endseq" → [1, 4, 5, 6, 3]


# === Calculate vocabulary sizes ===

# Count unique words in English
# +1 accounts for unknown words (words not in training data)
input_vocab_size = len(input_tokenizer.word_index) + 1
# Example: If English has 20 unique words, input_vocab_size = 21

# Count unique words in Spanish
# +1 for unknown words
output_vocab_size = len(output_tokenizer.word_index) + 1
# Example: If Spanish has 25 unique words, output_vocab_size = 26

# These sizes are crucial:
# - Embedding layer needs to know vocab size
# - Final Dense layer output will be this size (one score per possible word)
```

### 2.3 Padding Sequences (Make All Sequences Same Length)

```python
# Find the longest sequence length for inputs and outputs
# This ensures we know how much padding is needed

# max_input_length: Length of longest English sentence
max_input_length = max([len(seq) for seq in input_sequences])
# Example: If sequences are [1], [2,3,4], [5,6]
# Then max_input_length = 3

# max_output_length: Length of longest Spanish sentence  
max_output_length = max([len(seq) for seq in output_sequences])
# Example: If sequences are [1,2,3], [1,4,5,6,7,3]
# Then max_output_length = 6


# Pad all input sequences to the same length
# Shorter sequences get zeros added at the end
input_sequences = pad_sequences(
    input_sequences,           # Input: sequences of varying lengths
    maxlen=max_input_length,   # Target length: pad to this length
    padding='post'             # Add zeros at the END (not beginning)
)

# Example transformation:
# Before: [1], [2, 3], [4, 5, 6]
# After:  [1, 0, 0], [2, 3, 0], [4, 5, 6]  (all length 3)


# Same for output sequences
output_sequences = pad_sequences(
    output_sequences,
    maxlen=max_output_length,
    padding='post'
)

# Why padding?
# - Neural networks expect fixed input sizes
# - Padding with zeros doesn't affect meaning (zeros are neutral)
# - Later we'll use masking to ignore padded positions
```

### 2.4 Prepare Decoder Input and Output Data

```python
# The decoder learns by predicting the NEXT word given previous words
# So we need to shift the sequence: use [word1, word2, word3] 
# to predict [word2, word3, word4]

# Create decoder INPUT: all words except the LAST
decoder_input_data = output_sequences[:, :-1]
# :-1 means "all columns except last one"
# Example: [1, 2, 3, 4, 5] → [1, 2, 3, 4]
# (remove the final "endseq" token)

# Create decoder OUTPUT: all words except the FIRST  
decoder_output_data = output_sequences[:, 1:]
# 1: means "from column 1 onwards"
# Example: [1, 2, 3, 4, 5] → [2, 3, 4, 5]
# (remove the initial "startseq" token)

# Training works like this:
# Input:  [startseq, word1, word2]
# Output: [word1, word2, word3]  ← predict what comes next


# Convert decoder output to one-hot encoding
# One-hot: Vector with single 1 and rest 0s
# Example: word_index=2, vocab_size=5 → [0, 0, 1, 0, 0]
#                                              ↑ position 2

decoder_output_data = np.array([
    # For each sequence in decoder_output_data
    np.eye(output_vocab_size)[seq]  # Convert sequence to one-hot
    # np.eye creates identity matrix (basis vectors)
    # Index [seq] selects appropriate rows to create one-hot encoding
    for seq in decoder_output_data
])

# Why one-hot?
# - categorical_crossentropy loss requires one-hot encoded targets
# - Makes it a classification problem (choosing 1 word from vocab)
# - Loss = -log(predicted_probability_of_correct_word)

# Final shape:
# decoder_input_data: (num_samples, sequence_length)
# decoder_output_data: (num_samples, sequence_length, vocab_size)
```

---

## SECTION 3: Self-Attention Layer

### Understanding Self-Attention Mechanism

```
Self-Attention answers: "For each word, which other words should I focus on?"

Example: "The cat sat on the mat"
When processing "cat":
- The: 30% attention (article modifying cat)
- cat: 35% attention (itself, most relevant)
- sat: 25% attention (action of cat)
- on: 5% attention (not very relevant)
- the: 5% attention (preposition)
- mat: 0% attention (separate concept)

The model learns these percentages during training!
```

### Custom SelfAttention Layer Class

```python
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K

# Create a custom layer by inheriting from Keras Layer class
class SelfAttention(Layer):
    # Inheritance means it gets all Layer functionality
    # But we customize it for our attention mechanism
    
    def __init__(self, **kwargs):
        # **kwargs: accepts any keyword arguments (standard for custom layers)
        super(SelfAttention, self).__init__(**kwargs)
        # super() calls parent class __init__ to properly initialize Layer
```

### Build Method (Initialize Trainable Weights)

```python
    def build(self, input_shape):
        # input_shape: tuple describing shape of input
        # For attention: input_shape is list of 3 shapes [q_shape, k_shape, v_shape]
        # Each shape is (batch_size, sequence_length, feature_dim)
        
        # Extract feature dimension (usually 256 in our model)
        # input_shape[0][-1] means: first input, last dimension
        feature_dim = input_shape[0][-1]
        
        # Create weight matrix for Query (Q)
        # Used to transform input into "what we're looking for"
        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),  # 256 × 256 matrix
            initializer='glorot_uniform',      # Random initialization (good for learning)
            trainable=True,                    # This weight will be updated during training
            name='Wq'                          # Name for debugging
        )
        
        # Create weight matrix for Key (K)
        # Used to transform input into "what each word represents"
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wk'
        )
        
        # Create weight matrix for Value (V)
        # Used to transform input into "actual word content"
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wv'
        )
        
        # Call parent class build method
        super(SelfAttention, self).build(input_shape)
```

### Call Method (The Actual Attention Computation)

```python
    def call(self, inputs):
        # inputs: list of [query, key, value] tensors
        # Unpack the three components
        q, k, v = inputs
        
        # Transform inputs using weight matrices
        q = K.dot(q, self.Wq)  # Query = input @ Wq
        k = K.dot(k, self.Wk)  # Key = input @ Wk
        v = K.dot(v, self.Wv)  # Value = input @ Wv
        
        # Q, K, V all have shape: (batch_size, sequence_length, 256)
        
        
        # ===== STEP 1: Calculate Attention Scores =====
        # Scores show similarity between queries and keys
        # dot product = similarity measure
        
        scores = K.batch_dot(q, k, axes=[2, 2])
        # axes=[2, 2]: multiply along dimensions 2 of both q and k
        # batch_dot: performs batched matrix multiplication
        # Result shape: (batch_size, seq_length, seq_length)
        # Element [i,j] = how much word i attends to word j
        
        # Example: "The cat" attention scores
        # scores[0] = [0.5, 2.1]  (word 1 "The" attends to: itself 0.5, "cat" 2.1)
        # scores[1] = [0.3, 1.8]  (word 2 "cat" attends to: "The" 0.3, itself 1.8)
        
        
        # ===== STEP 2: Scale the Scores =====
        # Scaling prevents scores from becoming too large
        # Large values break softmax function
        
        dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
        # Get feature dimension size (256) and convert to float
        
        scores = scores / K.sqrt(dk)
        # Divide by sqrt(256) = 16
        # This normalization helps training stability
        
        # Scaled scores: [0.5/16, 2.1/16] = [0.031, 0.131]
        
        
        # ===== STEP 3: Apply Softmax =====
        # Softmax converts scores to probabilities
        # Output: numbers between 0 and 1 that sum to 1
        
        attention_weights = K.softmax(scores, axis=-1)
        # axis=-1: apply softmax along last axis
        
        # Example: [0.031, 0.131] → [0.35, 0.65]
        # Interpretation: 35% attention to first word, 65% to second word
        
        # Shape: (batch_size, seq_length, seq_length)
        # Each row sums to 1.0 (valid probability distribution)
        
        
        # ===== STEP 4: Apply Weights to Values =====
        # Use attention weights to create weighted sum of value vectors
        
        output = K.batch_dot(attention_weights, v)
        # attention_weights shape: (batch, seq_len, seq_len)
        # v shape: (batch, seq_len, 256)
        # output shape: (batch, seq_len, 256)
        
        # For each query position:
        # output[i] = weight[i,0]*v[0] + weight[i,1]*v[1] + ... 
        # High weight = that value contributes more to output
        
        return output
        # Return: context-aware representation
        # Same shape as input but with cross-word information encoded
```

---

## SECTION 4: Model Architecture

### Building the Encoder

```python
# ===== ENCODER: Understand English =====

# Define input layer
# Expects sequences of integers (tokenized words)
encoder_inputs = Input(shape=(max_input_length,))
# shape=(max_input_length,): single integer sequence per sample
# Example: (batch_size, 5) where 5 = max sentence length

# Convert word integers to dense vectors (embeddings)
# Each word number becomes a 256-dimensional vector
encoder_embedding = Embedding(
    input_vocab_size,      # Vocabulary size: 21 in our example
    256                    # Embedding dimension: 256-d vectors
)(encoder_inputs)

# Result shape: (batch_size, sequence_length, 256)
# Example: (16, 5, 256) - 16 samples, max 5 words, each 256-d

# Create LSTM layer to process sequences
# LSTM: processes word vectors one by one, maintaining context
encoder_lstm = LSTM(
    256,                   # Number of hidden units (memory cells)
    return_sequences=True, # Return output for EACH time step (each word)
    return_state=True      # Also return final hidden and cell states
)

# Run LSTM on embeddings
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# Breakdown of outputs:
# encoder_outputs: All hidden states (one per word)
#   Shape: (batch_size, sequence_length, 256)
#   Contains: memory/context after processing each word
#
# state_h: Final hidden state (after processing all words)
#   Shape: (batch_size, 256)
#   Contains: compressed understanding of entire sentence
#
# state_c: Final cell state (long-term memory)
#   Shape: (batch_size, 256)
#   Contains: long-term patterns learned

# Store final states to pass to decoder
encoder_states = [state_h, state_c]
# These states "tell" the decoder what the encoder learned
```

### Building the Decoder

```python
# ===== DECODER: Generate Spanish =====

# Define input layer for Spanish
# Shifted sequences (one step back, as explained earlier)
decoder_inputs = Input(shape=(max_output_length - 1,))
# max_output_length - 1: removed endseq token

# Convert Spanish word integers to embeddings
decoder_embedding = Embedding(
    output_vocab_size,  # Spanish vocabulary size: 26 in our example
    256                 # Embedding dimension: 256-d vectors
)(decoder_inputs)

# Result shape: (batch_size, sequence_length, 256)

# Create LSTM for decoder
decoder_lstm = LSTM(
    256,                   # Same hidden dimension as encoder (important for state transfer!)
    return_sequences=True, # Return all hidden states
    return_state=True
)

# Run LSTM with encoder's final states as initial states
# This is KEY: decoder "initializes" with encoder's understanding!
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states  # Start from encoder's final state
)

# Breakdown:
# decoder_outputs: All hidden states from decoder LSTM
#   Shape: (batch_size, sequence_length, 256)
#   Contains: context for predicting each Spanish word
#
# _, _: We ignore final states (don't need them for prediction)
```

### Adding Self-Attention Layer

```python
# ===== ATTENTION: Focus on Relevant English Words =====

# Create an instance of our custom SelfAttention layer
self_attention = SelfAttention()

# Apply attention
# Query: "What Spanish words do I need to generate?"
# Key: "What information is in the English sentence?"
# Value: "Which English words should I look at?"
attention_output = self_attention([
    decoder_outputs,      # Q: Spanish context (what to generate)
    encoder_outputs,      # K: English context (what's available)
    encoder_outputs       # V: English information (what to use)
])

# Why decoder_outputs as query?
# The decoder "queries" the encoder: "Hey encoder, what do I need to know 
# to generate this Spanish word?"

# Why encoder_outputs for K and V?
# The encoder "answers": "Here's what I know about the English sentence.
# These are my key findings (K) and my detailed information (V)."

# Result shape: (batch_size, sequence_length, 256)
# Contains: English context relevant to each Spanish position
```

### Combining Decoder Outputs with Attention

```python
# Decoder output has information about Spanish translation
# Attention output has information about English source
# Combine both for better predictions!

decoder_concat = Concatenate(axis=-1)([
    decoder_outputs,    # Spanish translation context
    attention_output    # English source context
])

# Concatenate axis=-1: append along feature dimension
# Before concat: each of shape (batch, seq_len, 256)
# After concat: shape (batch, seq_len, 512)  [256+256=512]

# Why combine?
# Decoder knows: "I'm translating 'cat'"
# Attention knows: "Focus on words 'the', 'sat' from English"
# Together they know: "Translate 'cat' considering what it refers to in English"
```

### Final Prediction Layer

```python
# Convert combined context to word probabilities
decoder_dense = Dense(
    output_vocab_size,    # Output: probability for each Spanish word
    activation='softmax'  # Convert to probabilities (0-1, sum to 1)
)

# Apply Dense layer
decoder_outputs = decoder_dense(decoder_concat)

# Process:
# Input shape: (batch_size, sequence_length, 512)
# 1. Dense layer: 512 → output_vocab_size (e.g., 512 → 26)
# 2. Softmax: converts to probabilities
# 3. Output shape: (batch_size, sequence_length, 26)

# Result meaning:
# decoder_outputs[sample, position, :] = probability distribution over Spanish words
# Example: [0.05, 0.8, 0.1, 0.05] 
# = 5% prob of word1, 80% prob of word2 (likely "Hola"), 10% word3, 5% word4

# argmax of this gives: the PREDICTED word
# max value (0.8) is the CONFIDENCE in prediction
```

### Link All Components into a Model

```python
# Create the complete model
model = Model(
    [encoder_inputs, decoder_inputs],  # Inputs: English and Spanish (shifted)
    decoder_outputs                     # Output: predicted Spanish words
)

# The Model class links all layers:
# English → Encoder → Decoder → Attention → Dense → Spanish
#            ↓
#   Stored states (h, c)
#            ↓
#   Used as initial state

# Compile: prepare for training
model.compile(
    optimizer='adam',                    # Update rule for weights
    loss='categorical_crossentropy',     # How to measure error
    metrics=['accuracy']                 # Evaluate accuracy
)

# optimizer='adam':
# Adam = Adaptive Moment Estimation
# Automatically adjusts learning rate for each weight
# Usually better than fixed learning rate

# loss='categorical_crossentropy':
# For multi-class classification (choose one word from vocabulary)
# Formula: -sum(true_one_hot * log(predicted_probs))
# Low loss when model predicts correct word with high probability

# metrics=['accuracy']:
# Track: percentage of predictions matching target
# Example: 80% accuracy = model gets 4 out of 5 words correct
```

### Inspect Model

```python
# Print model summary
model.summary()
# Shows:
# - All layers and their configurations
# - Number of parameters (weights to train)
# - Input/output shapes
# - Total trainable parameters
```

---

## SECTION 5: Training

```python
# ===== TRAIN THE MODEL =====

# Fit model to training data
history = model.fit(
    [input_sequences, decoder_input_data],  # Inputs
    decoder_output_data,                     # Target outputs
    epochs=100,                              # Train for 100 passes through data
    batch_size=16                            # Process 16 samples at a time
)

# What happens in model.fit():

# FOR epoch = 1 to 100:
#   FOR each batch of 16 samples:
#     1. FORWARD PASS
#        - Feed English sentence through encoder
#        - Feed Spanish (shifted) through decoder
#        - Get predictions (probabilities for each word)
#
#     2. CALCULATE LOSS
#        - Compare predictions to actual Spanish words
#        - Loss = -log(probability of correct word)
#        - Lower loss = better prediction
#
#     3. BACKWARD PASS (Backpropagation)
#        - Calculate gradient (rate of change) for each weight
#        - Shows: which direction to adjust weights
#        - How much adjustment needed
#
#     4. UPDATE WEIGHTS (Optimization)
#        - Move weights in direction of lower loss
#        - Amount to move = learning_rate × gradient
#        - Adam optimizer adjusts learning_rate per parameter
#   END

# history: contains training metrics
# history.history['loss']: list of loss values per epoch
# Can be used to plot training progress
```

---

## SECTION 6: Visualizing Training

```python
import matplotlib.pyplot as plt

# Plot training loss over epochs
plt.plot(history.history['loss'])  # Get loss values from history
plt.title('Training Loss')         # Chart title
plt.xlabel('Epochs')               # X-axis label
plt.ylabel('Loss')                 # Y-axis label
plt.show()

# Good signs in the plot:
# - Loss line goes DOWN (model learning)
# - Smooth curve (stable training)
# - Eventually plateaus (converged)

# Bad signs:
# - Loss goes UP (overfitting)
# - Very noisy (unstable)
# - Stays flat (not learning)

# Interpretation:
# Epoch 1: Loss = 5.2   (bad predictions)
# Epoch 10: Loss = 3.1  (improving)
# Epoch 50: Loss = 0.8  (good predictions)
# Epoch 100: Loss = 0.3 (very good predictions)
```

---

## SECTION 7: Practice Exercises Explained

### Exercise 1: Different Weight Initializers

```python
# Original code used 'glorot_uniform' initializer
# This exercise uses 'he_uniform' instead

# In SelfAttention class, change:
# OLD: initializer='glorot_uniform'
# NEW: initializer='he_uniform'

# Why different initializers?
# Different random starting weights → different learning curves
# Some initializers help networks learn faster
# This exercise compares which is better for our task

# Expected outcome:
# Graph shows two loss curves side by side
# One might decrease faster (better initializer)
```

### Exercise 2: Different Optimizers

```python
# Original code used 'adam' optimizer
# This exercise uses 'adagrad' instead

# In model.compile(), change:
# OLD: optimizer='adam'
# NEW: optimizer='adagrad'

# Why different optimizers?
# Each optimizer has different update rule
# Adam: adaptive per-weight, momentum
# Adagrad: adaptive per-weight, accumulates gradient history

# Adam usually faster, Adagrad good for sparse data
# This exercise shows which works better

# Expected outcome:
# Graph shows adam faster than adagrad
# Or vice versa depending on data
```

---

## Complete Code Structure Summary

```
1. IMPORTS
   ├── Libraries for neural networks (keras)
   ├── Text processing (tokenizer)
   └── Math operations (numpy, tensorflow)

2. DATA PREPARATION
   ├── Define sentences (English-Spanish pairs)
   ├── Tokenize (words → numbers)
   ├── Pad (all same length)
   └── One-hot encode (prepare for loss function)

3. CUSTOM LAYER
   └── SelfAttention
       ├── Build: Initialize weight matrices (Wq, Wk, Wv)
       └── Call: Compute attention
           ├── Calculate scores (Q·K^T)
           ├── Scale by √dk
           ├── Softmax to probabilities
           └── Apply to values

4. MODEL ARCHITECTURE
   ├── Encoder
   │   ├── Embedding: words → vectors
   │   └── LSTM: learn sequence patterns
   │
   ├── Decoder
   │   ├── Embedding: words → vectors
   │   ├── LSTM: generate translation (initialized with encoder state)
   │   └── Uses encoder state as "context"
   │
   ├── Attention: decoder looks at encoder
   │
   └── Output
       ├── Concatenate: combine decoder + attention
       └── Dense + Softmax: predict word probabilities

5. COMPILE
   ├── Optimizer: adam (how to update weights)
   ├── Loss: categorical_crossentropy (measure error)
   └── Metrics: accuracy (track performance)

6. TRAIN
   └── For 100 epochs:
       ├── Forward pass (predictions)
       ├── Calculate loss
       ├── Backward pass (gradients)
       └── Update weights

7. VISUALIZE
   └── Plot loss over epochs
```

This structure follows the typical deep learning pipeline:
Define → Prepare Data → Build Model → Compile → Train → Evaluate

---

## Key Equations at a Glance

```
EMBEDDING:
word_index → embedding_vector
e.g., 2 → [0.1, 0.3, -0.2, ..., 0.5]  (256 dimensions)

ATTENTION SCORES:
scores = Q · K^T          (matrix multiplication)
scores = scores / √d_k    (scaling)

ATTENTION WEIGHTS:
weights = softmax(scores)  (convert to probabilities)

ATTENTION OUTPUT:
output = weights · V       (weighted sum of values)

LOSS (Categorical Crossentropy):
loss = -log(predicted_prob_of_correct_word)
Example:
- Correct word: "Hola"
- Model predicts: [0.05, 0.8, 0.1, 0.05] (80% for position 2)
- Loss = -log(0.8) ≈ 0.22 (low loss = good prediction)

WEIGHT UPDATE (Gradient Descent with Adam):
new_weight = old_weight - learning_rate × gradient
(adam adjusts learning_rate per parameter)
```

---

## Debugging Tips

### If Loss is Not Decreasing:
1. Learning rate too high → Use smaller learning_rate
2. Learning rate too low → Use larger learning_rate
3. Model too simple → Add more units/layers
4. Data problem → Check input/output data shape and content
5. Bug in code → Print shapes at each step

### If Loss Becomes NaN:
1. Learning rate too high → Reduce learning_rate
2. Gradient exploding → Use gradient clipping
3. Data problem → Check for Infinity or NaN in data

### To Debug Shapes:
```python
print(f"Input shape: {input_sequences.shape}")
print(f"Encoder output shape: {encoder_outputs.shape}")
print(f"Attention output shape: {attention_output.shape}")
print(f"Final output shape: {decoder_outputs.shape}")
```

### To Check Data:
```python
print(f"Sample input: {input_sequences[0]}")
print(f"Sample target: {decoder_output_data[0]}")
print(f"Loss value: {history.history['loss'][-1]}")
```

This detailed explanation covers every line of the notebook!
